# CUDA on Colab
a register-blocked (2D thread-tiling) kernel where each thread computes a TM×TN micro-tile of C in registers, with two configs (4×4 and 8×8) as a mini-sweep.

In [1]:
%%writefile matmul_cuda.cu
/* matmul_cuda.cu -- GPU dense matrix multiplication (HPC project, Phase 5, Colab T4)
 *   C = A x B,  A=2, B=3  ->  every C[i][j] == 6*n  (oracle max|c-6n|).
 *   Progression (FP32 + FP64): naive -> shared-tiled -> register-blocked -> cuBLAS.
 *   Build: nvcc -O3 -arch=sm_75 matmul_cuda.cu -o matmul_cuda -lcublas
 */
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CK(call) do { cudaError_t _ck=(call); if(_ck!=cudaSuccess){ \
    fprintf(stderr,"CUDA error %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(_ck)); exit(1);}}while(0)

template <typename T>
__global__ void matmul_naive(const T* A, const T* B, T* C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        T acc = 0;
        for (int k = 0; k < n; ++k) acc += A[row * n + k] * B[k * n + col];
        C[row * n + col] = acc;
    }
}

template <typename T, int TILE>
__global__ void matmul_tiled(const T* A, const T* B, T* C, int n) {
    __shared__ T As[TILE][TILE];
    __shared__ T Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    T acc = 0;
    for (int t = 0; t < (n + TILE - 1) / TILE; ++t) {
        int aCol = t * TILE + threadIdx.x;
        int bRow = t * TILE + threadIdx.y;
        As[threadIdx.y][threadIdx.x] = (row < n && aCol < n) ? A[row * n + aCol] : (T)0;
        Bs[threadIdx.y][threadIdx.x] = (bRow < n && col < n) ? B[bRow * n + col] : (T)0;
        __syncthreads();
        for (int k = 0; k < TILE; ++k) acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }
    if (row < n && col < n) C[row * n + col] = acc;
}

/* Register-blocked (2D thread-tiling): each thread computes a TM x TN micro-tile of C
   in registers (GotoBLAS/CUTLASS style). Bounds via zero-pad loads + guarded stores. */
template <typename T, int BM, int BN, int BK, int TM, int TN>
__global__ void matmul_reg(const T* A, const T* B, T* C, int n) {
    __shared__ T As[BM][BK];
    __shared__ T Bs[BK][BN];
    const int blockRow = blockIdx.y * BM, blockCol = blockIdx.x * BN;
    const int threadCol = threadIdx.x, threadRow = threadIdx.y;
    const int nThreads = (BM / TM) * (BN / TN);
    const int tid = threadRow * (BN / TN) + threadCol;
    T acc[TM][TN];
    #pragma unroll
    for (int i = 0; i < TM; ++i)
        #pragma unroll
        for (int j = 0; j < TN; ++j) acc[i][j] = (T)0;
    for (int kk = 0; kk < n; kk += BK) {
        for (int idx = tid; idx < BM * BK; idx += nThreads) {
            int r = idx / BK, c = idx % BK, gr = blockRow + r, gc = kk + c;
            As[r][c] = (gr < n && gc < n) ? A[(size_t)gr * n + gc] : (T)0;
        }
        for (int idx = tid; idx < BK * BN; idx += nThreads) {
            int r = idx / BN, c = idx % BN, gr = kk + r, gc = blockCol + c;
            Bs[r][c] = (gr < n && gc < n) ? B[(size_t)gr * n + gc] : (T)0;
        }
        __syncthreads();
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            T aReg[TM], bReg[TN];
            #pragma unroll
            for (int i = 0; i < TM; ++i) aReg[i] = As[threadRow * TM + i][k];
            #pragma unroll
            for (int j = 0; j < TN; ++j) bReg[j] = Bs[k][threadCol * TN + j];
            #pragma unroll
            for (int i = 0; i < TM; ++i)
                #pragma unroll
                for (int j = 0; j < TN; ++j) acc[i][j] += aReg[i] * bReg[j];
        }
        __syncthreads();
    }
    #pragma unroll
    for (int i = 0; i < TM; ++i) {
        int gr = blockRow + threadRow * TM + i;
        #pragma unroll
        for (int j = 0; j < TN; ++j) {
            int gc = blockCol + threadCol * TN + j;
            if (gr < n && gc < n) C[(size_t)gr * n + gc] = acc[i][j];
        }
    }
}

static void gemm(cublasHandle_t h, int n, const float* A, const float* B, float* C) {
    float a = 1.f, b = 0.f;
    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, n, n, n, &a, B, n, A, n, &b, C, n);
}
static void gemm(cublasHandle_t h, int n, const double* A, const double* B, double* C) {
    double a = 1.0, b = 0.0;
    cublasDgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, n, n, n, &a, B, n, A, n, &b, C, n);
}

template <typename T>
static double check(const T* C, int n) {
    double expected = 6.0 * n, maxe = 0.0;
    for (size_t t = 0; t < (size_t)n * n; ++t) { double e = fabs((double)C[t] - expected); if (e > maxe) maxe = e; }
    return maxe;
}
static float time_ms(cudaEvent_t s, cudaEvent_t e) { float ms; cudaEventElapsedTime(&ms, s, e); return ms; }

template <typename T>
static void run(int n, int reps, const char* tname) {
    size_t bytes = (size_t)n * n * sizeof(T);
    T *hA = (T*)malloc(bytes), *hB = (T*)malloc(bytes), *hC = (T*)malloc(bytes);
    for (size_t t = 0; t < (size_t)n * n; ++t) { hA[t] = (T)2; hB[t] = (T)3; }
    T *dA, *dB, *dC;
    CK(cudaMalloc(&dA, bytes)); CK(cudaMalloc(&dB, bytes)); CK(cudaMalloc(&dC, bytes));
    cudaEvent_t s, e; CK(cudaEventCreate(&s)); CK(cudaEventCreate(&e));
    CK(cudaEventRecord(s));
    CK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e));
    float h2d = time_ms(s, e);
    const double gf = 2.0 * (double)n * n * n / 1e9;
    auto finish = [&](const char* name, float best_ms) {
        CK(cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost));
        printf("cuda %-6s %-14s n=%d | kernel=%.3f ms | %.2f GFLOP/s | H2D=%.3f ms | max_err=%.3g\n",
               tname, name, n, best_ms, gf / (best_ms / 1e3), h2d, check(hC, n));
    };
    for (int bsz : {8, 16, 32}) {
        dim3 blk(bsz, bsz), grd((n + bsz - 1) / bsz, (n + bsz - 1) / bsz);
        float best = 1e30f;
        for (int r = 0; r < reps; ++r) { CK(cudaEventRecord(s)); matmul_naive<T><<<grd, blk>>>(dA, dB, dC, n);
            CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); CK(cudaGetLastError()); best = fminf(best, time_ms(s, e)); }
        char nm[24]; snprintf(nm, sizeof nm, "naive-b%d", bsz); finish(nm, best);
    }
    { dim3 blk(16,16), grd((n+15)/16,(n+15)/16); float best=1e30f;
      for(int r=0;r<reps;++r){ CK(cudaEventRecord(s)); matmul_tiled<T,16><<<grd,blk>>>(dA,dB,dC,n);
        CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); CK(cudaGetLastError()); best=fminf(best,time_ms(s,e)); } finish("tiled-16",best); }
    { dim3 blk(32,32), grd((n+31)/32,(n+31)/32); float best=1e30f;
      for(int r=0;r<reps;++r){ CK(cudaEventRecord(s)); matmul_tiled<T,32><<<grd,blk>>>(dA,dB,dC,n);
        CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); CK(cudaGetLastError()); best=fminf(best,time_ms(s,e)); } finish("tiled-32",best); }
    { dim3 blk(16,16), grd((n+63)/64,(n+63)/64); float best=1e30f;
      for(int r=0;r<reps;++r){ CK(cudaEventRecord(s)); matmul_reg<T,64,64,8,4,4><<<grd,blk>>>(dA,dB,dC,n);
        CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); CK(cudaGetLastError()); best=fminf(best,time_ms(s,e)); } finish("reg-4x4",best); }
    { dim3 blk(16,16), grd((n+127)/128,(n+127)/128); float best=1e30f;
      for(int r=0;r<reps;++r){ CK(cudaEventRecord(s)); matmul_reg<T,128,128,8,8,8><<<grd,blk>>>(dA,dB,dC,n);
        CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); CK(cudaGetLastError()); best=fminf(best,time_ms(s,e)); } finish("reg-8x8",best); }
    { cublasHandle_t h; cublasCreate(&h); float best=1e30f;
      for(int r=0;r<reps;++r){ CK(cudaEventRecord(s)); gemm(h,n,dA,dB,dC);
        CK(cudaEventRecord(e)); CK(cudaEventSynchronize(e)); best=fminf(best,time_ms(s,e)); } cublasDestroy(h); finish("cublas",best); }
    cudaEventDestroy(s); cudaEventDestroy(e);
    cudaFree(dA); cudaFree(dB); cudaFree(dC); free(hA); free(hB); free(hC);
}

int main(int argc, char** argv) {
    if (argc < 2) { fprintf(stderr, "Usage: %s n [reps]\n", argv[0]); return 1; }
    int n = atoi(argv[1]); int reps = (argc > 2) ? atoi(argv[2]) : 3;
    if (n <= 0 || reps <= 0) { fprintf(stderr, "invalid arguments\n"); return 1; }
    int dev = 0; cudaDeviceProp p; CK(cudaGetDevice(&dev)); CK(cudaGetDeviceProperties(&p, dev));
    printf("# GPU: %s  CC %d.%d  SMs=%d  globalMem=%.1f GB\n", p.name, p.major, p.minor, p.multiProcessorCount, p.totalGlobalMem/1e9);
    printf("# reps=%d\n\n", reps);
    printf("=== FP32 (float) ===\n");   run<float>(n, reps, "fp32");
    printf("\n=== FP64 (double) ===\n"); run<double>(n, reps, "fp64");
    return 0;
}

Writing matmul_cuda.cu


In [2]:
!nvcc -O3 -arch=sm_75 matmul_cuda.cu -o matmul_cuda -lcublas
!./matmul_cuda 5000  2>&1 | tee cuda_5000_reg.txt
!./matmul_cuda 10000 2>&1 | tee cuda_10000_reg.txt
from google.colab import files
files.download('cuda_5000_reg.txt'); files.download('cuda_10000_reg.txt')

# GPU: Tesla T4  CC 7.5  SMs=40  globalMem=15.6 GB
# reps=3

=== FP32 (float) ===
cuda fp32   naive-b8       n=5000 | kernel=937.842 ms | 266.57 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   naive-b16      n=5000 | kernel=520.121 ms | 480.66 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   naive-b32      n=5000 | kernel=422.090 ms | 592.29 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   tiled-16       n=5000 | kernel=358.374 ms | 697.60 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   tiled-32       n=5000 | kernel=276.737 ms | 903.38 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   reg-4x4        n=5000 | kernel=100.835 ms | 2479.30 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   reg-8x8        n=5000 | kernel=74.745 ms | 3344.71 GFLOP/s | H2D=43.583 ms | max_err=0
cuda fp32   cublas         n=5000 | kernel=55.735 ms | 4485.54 GFLOP/s | H2D=43.583 ms | max_err=0

=== FP64 (double) ===
cuda fp64   naive-b8       n=5000 | kernel=1684.579 ms | 148.41 GFLOP/s | H2D=84.623 ms | max_err=0
cu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>